# Lesson 10 · AI Agents I · Demo
## Um agente do zero, chamada por chamada

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

> *"Quero um SUV para a família, até uns 80 mil. O que você recomenda?"*

O mesmo pedido, o mesmo modelo (`qwen2.5:3b`, via Ollama) e o mesmo *system prompt*, em dois exemplos.

| | O que o modelo recebe | O que acontece |
|---|---|---|
| **Exemplo 1** | só a conversa | ele não tem como olhar o estoque |
| **Exemplo 2** | a conversa **e a descrição de três ferramentas** (buscar, detalhar, comparar) | ele pede uma ferramenta, nós executamos, ele lê o resultado e responde |

No Exemplo 2, cada chamada intermediária aparece na tela: o que vai para o modelo, o JSON que ele devolve, a função que nós executamos e o resultado que volta para ele. No fim, essas peças viram o loop completo do agente.

Sem framework nenhum. Só `requests`, `json` e `pandas`.

**Como rodar.** No Colab, escolha o runtime **T4 GPU** (Ambiente de execução → Alterar o tipo de ambiente de execução) e rode tudo de cima a baixo.

---
## 🧰 Preparação

A célula abaixo confere cada passo e só faz o que falta. Instala o Ollama se ele não existir (no Colab), sobe o servidor se ele não estiver de pé e baixa o `qwen2.5:3b` (~2 GB) se ele ainda não foi baixado. Pode rodar quantas vezes quiser. Na primeira vez no Colab, leva uns 3 minutos.

In [1]:
# @title 🧰 Preparação: confere o Ollama, o servidor e o modelo, e só instala o que faltar
import os, shutil, subprocess, sys, time, requests

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"
IN_COLAB = "google.colab" in sys.modules
sh = get_ipython().system                      # roda um comando de shell mostrando a saída na célula

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

# 1 · O programa ollama existe nesta máquina?
if shutil.which("ollama"):
    print("✅ Ollama já instalado")
elif IN_COLAB:
    print("⏳ instalando o Ollama (cerca de 1 minuto)...")
    sh("apt-get install -y -qq zstd > /dev/null 2>&1 || "
       "(apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)")
    sh("curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2")
    if not shutil.which("ollama"):
        raise RuntimeError("a instalação falhou. Rode a célula de novo.")
    print("✅ Ollama instalado")
else:
    raise RuntimeError("Ollama não encontrado. Instale em https://ollama.com/download e rode esta célula de novo.")

# 2 · O servidor está respondendo?
if server_is_up():
    print("✅ servidor já está de pé")
else:
    print("⏳ subindo o servidor...")
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)  # continua vivo depois que a célula termina
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
    if not server_is_up():
        raise RuntimeError("o servidor não respondeu. Veja o arquivo ollama.log")
    print("✅ servidor de pé")

# 3 · O modelo já foi baixado?
installed = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
if MODEL in installed:
    print(f"✅ {MODEL} já baixado")
else:
    print(f"⏳ baixando {MODEL} (~2 GB). Se a conexão cair, rode a célula de novo: o download continua.")
    sh(f"ollama pull {MODEL}")
    print(f"✅ {MODEL} baixado")

# 4 · Pacotes Python. O Colab já traz todos; só instala se faltar algum.
try:
    import pandas
except ImportError:
    sh("pip -q install pandas requests")

version = requests.get(f"{OLLAMA_URL}/api/version").json()["version"]
print(f"\n🚀 pronto · Ollama {version} · modelo {MODEL} · {'Colab' if IN_COLAB else 'máquina local'}")

⏳ instalando o Ollama (cerca de 1 minuto)...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama instalado
⏳ subindo o servidor...
✅ servidor de pé
⏳ baixando qwen2.5:3b (~2 GB). Se a conexão cair, rode a célula de novo: o download continua.

✅ qwen2.5:3b baixado

🚀 pronto · Ollama 0.40.11 · modelo qwen2.5:3b · Colab


In [2]:
# @title 🔧 Utilitários: a chamada ao modelo e duas funções para imprimir a conversa
import json, re
import pandas as pd

def llm(messages, tools=None):
    """Uma chamada ao /api/chat do Ollama. Devolve o JSON inteiro da resposta."""
    body = {"model": MODEL, "messages": messages, "stream": False,
            "options": {"temperature": 0,          # zero: a mesma entrada dá (quase) a mesma saída
                        "num_predict": 600}}       # teto de tokens por resposta, para nada travar
    if tools:
        body["tools"] = tools
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=body, timeout=300)
    r.raise_for_status()
    return r.json()

def show(obj, limit=1600):
    """Imprime um dicionário como JSON legível, cortado se for longo."""
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    print(text if len(text) <= limit else text[:limit] + "\n  ... (cortado)")

ICON = {"system": "⚙️ ", "user": "👤", "assistant": "🤖", "tool": "🔧"}

def show_messages(messages):
    """A conversa inteira, uma linha por mensagem."""
    for i, m in enumerate(messages):
        if m.get("tool_calls"):
            what = "pede " + " e ".join(f'{c["function"]["name"]}({json.dumps(c["function"]["arguments"], ensure_ascii=False)})'
                                        for c in m["tool_calls"])
        else:
            what = m["content"].replace("\n", " ")
        print(f"[{i}] {ICON[m['role']]} {m['role']:<9} │ {what[:120]}{'…' if len(what) > 120 else ''}")

print("utilitários prontos")

utilitários prontos


---
# Parte 1 · O estoque

São 320 anúncios de carros usados à venda em Natal e região. Os dados são sintéticos, mas os modelos, as faixas de preço e as cidades seguem o mercado de verdade. Antes de pôr qualquer modelo para trabalhar, respondemos o pedido à mão. Essa resposta é o **gabarito**.

In [4]:
# O estoque. Se o arquivo não estiver ao lado do notebook, baixa do repositório da disciplina.
CSV = "cars.csv"
if not os.path.exists(CSV):
    url = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week07/cars.csv"
    r = requests.get(url)
    r.raise_for_status()
    open(CSV, "wb").write(r.content)

cars = pd.read_csv(CSV)
print(f"{len(cars)} anúncios · colunas: {', '.join(cars.columns)}\n")

# O gabarito: SUVs até R$ 80 mil, dos mais novos para os mais velhos (e, no mesmo ano, os menos rodados).
suvs = cars[(cars.body == "SUV") & (cars.price_brl <= 80_000)].sort_values(["year", "km"], ascending=[False, True])
print(f"{len(suvs)} SUVs até R$ 80 mil. Os três mais novos:")
suvs[["car_id", "brand", "model", "year", "km", "price_brl", "trunk_l"]].head(3)

320 anúncios · colunas: car_id, brand, model, body, year, km, price_brl, transmission, fuel, consumption_kml, trunk_l, color, city

22 SUVs até R$ 80 mil. Os três mais novos:


,car_id,brand,model,year,km,price_brl,trunk_l
81,c082,Renault,Duster,2021,56500,75000,475
80,c081,Renault,Duster,2021,67700,71000,475
108,c109,Volkswagen,Nivus,2021,70300,75500,415


**O gabarito é `c082`, `c081` e `c109`**, dois Renault Duster 2021 e um VW Nivus 2021. Cada anúncio tem um `car_id`, e é por ele que vamos conferir as respostas do modelo.

---
# Parte 2 · Exemplo 1: sem ferramentas

O modelo recebe duas mensagens, o *system prompt* e o pedido. Repare que o *system prompt* já diz "nunca invente um carro" e "cite sempre o `car_id`". Ele é o mesmo nos dois exemplos.

🔮 **Antes de rodar.** O modelo não tem o arquivo `cars.csv`. O que ele vai responder?

In [5]:
SYSTEM = """You are a used-car buying assistant in Natal, Brazil. Answer in the user's language.
Use the tools to look up real listings; never invent a car.
Always cite the car_id of every car you mention."""

PEDIDO = "Quero um SUV para a família, até uns 80 mil. O que você recomenda?"

messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": PEDIDO}]

print("📤 O QUE ENVIAMOS AO MODELO")
show_messages(messages)

resp = llm(messages)                      # sem ferramentas
print("\n📥 O QUE ELE RESPONDEU\n")
print(resp["message"]["content"])

📤 O QUE ENVIAMOS AO MODELO
[0] ⚙️  system    │ You are a used-car buying assistant in Natal, Brazil. Answer in the user's language. Use the tools to look up real listi…
[1] 👤 user      │ Quero um SUV para a família, até uns 80 mil. O que você recomenda?

📥 O QUE ELE RESPONDEU

Para te ajudar com essa busca, precisarei pesquisar carros de usados no valor de até R$ 80.000 em Natal, Brasil. No entanto, como souza não tenho a capacidade de consultar diretamente o banco de dados de carros de usados, eu precisaria de sua ajuda para buscar essas informações. Você poderia fornecer um intervalo de datas para os carros que você está interessado, por exemplo, carros vendidos entre janeiro e dezembro deste ano? Isso ajudaria a filtrar a pesquisa de acordo com o seu interesse.


### 🔍 O que aconteceu

O modelo **descreve** como fazer a pesquisa ("use um site de classificados", "filtre por preço e tipo de veículo") e não faz nenhuma. Nenhum `car_id`, nenhum anúncio real. Ele obedeceu ao "nunca invente", mas sem acesso ao estoque não tem como ajudar. É o card "No hands" do slide. O modelo descreve ações e não executa nenhuma.

Sem a frase "never invent a car" no *system prompt*, o mesmo modelo costuma fazer o contrário e recomendar SUVs que nem estão no estoque, com toda a confiança. Nas duas versões o problema é o mesmo: **a única saída de um modelo é texto**. Para que esse texto tenha consequência, ele precisa de ferramentas.

---
# Parte 3 · Exemplo 2: com ferramentas, passo a passo

## 3.1 · As ferramentas são funções Python comuns

As três ferramentas dos slides, sem LLM nenhum: **buscar** no estoque, **detalhar** um carro e **comparar** carros. Testamos as três chamando direto, como qualquer função.

In [6]:
def search_cars(max_price, body=None):
    """Busca no estoque. Devolve quantos carros bateram no filtro e os 5 mais novos."""
    r = cars[cars.price_brl <= max_price]
    if body is not None:
        r = r[r.body == body]
    r = r.sort_values(["year", "km"], ascending=[False, True])
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission"]
    return {"total_found": len(r), "cars": r[cols].head(5).to_dict("records")}

def car_details(car_id):
    """Todos os atributos de um carro: porta-malas, consumo, cor, cidade..."""
    r = cars[cars.car_id == car_id]
    if r.empty:
        return {"error": f"no car with id {car_id}"}
    return r.to_dict("records")[0]

def compare_cars(car_ids):
    """Os carros lado a lado e, em cada critério, qual é o melhor (a conta é feita aqui, não pelo modelo)."""
    r = cars[cars.car_id.isin(car_ids)]
    if r.empty:
        return {"error": f"none of these ids exist: {car_ids}"}
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission", "trunk_l", "consumption_kml"]
    def winners(col, highest):
        target = r[col].max() if highest else r[col].min()
        return list(r[r[col] == target].car_id)              # mais de um id = empate
    best = {"largest_trunk": winners("trunk_l", True),
            "most_economical": winners("consumption_kml", True),   # mais km por litro = consome menos
            "cheapest": winners("price_brl", False),
            "newest": winners("year", True),
            "lowest_km": winners("km", False)}
    return {"cars": r[cols].to_dict("records"), "best": best,
            "missing": [i for i in car_ids if i not in set(r.car_id)]}

print("search_cars(80_000, body='SUV') →")
show(search_cars(80_000, body="SUV"))
print("\ncar_details('c082') →")
show(car_details("c082"))
print("\ncompare_cars(['c082', 'c109']) →")
show(compare_cars(["c082", "c109"]))

search_cars(80_000, body='SUV') →
{
  "total_found": 22,
  "cars": [
    {
      "car_id": "c082",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 56500,
      "price_brl": 75000,
      "transmission": "automatic"
    },
    {
      "car_id": "c081",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 67700,
      "price_brl": 71000,
      "transmission": "manual"
    },
    {
      "car_id": "c109",
      "brand": "Volkswagen",
      "model": "Nivus",
      "year": 2021,
      "km": 70300,
      "price_brl": 75500,
      "transmission": "manual"
    },
    {
      "car_id": "c061",
      "brand": "Jeep",
      "model": "Renegade",
      "year": 2020,
      "km": 82700,
      "price_brl": 76500,
      "transmission": "manual"
    },
    {
      "car_id": "c003",
      "brand": "Chevrolet",
      "model": "Tracker",
      "year": 2019,
      "km": 91000,
      "price_brl": 74500,
      "transmission": "automatic"
    }
  

Repare no campo `best` da comparação. A ferramenta já devolve, para cada critério, qual carro vence. Por que não deixar o modelo comparar sozinho? Porque, durante a preparação da aula, ele leu "Duster 10,4 km/l, Nivus 12,5 km/l" e concluiu que **o Duster consome menos**, que é o contrário (mais km por litro é mais econômico). É o card "To compute" do slide de ferramentas: LLMs adivinham contas, e ferramentas não. Quando houver empate, `best` traz os dois ids.

## 3.2 · O que o modelo recebe é a **descrição** das ferramentas

O modelo não tem como importar uma função Python. A cada chamada, ele recebe uma descrição de cada ferramenta, com **nome**, **descrição** (quando usar) e **parâmetros** (com tipos, num JSON Schema). É exatamente esta estrutura que vai na requisição.

In [7]:
TOOLS = [
    {"type": "function",
     "function": {
         "name": "search_cars",
         "description": ("Search the used-car stock by budget and body type. "
                         "Returns how many cars matched and the 5 newest ones (car_id, model, year, km, price)."),
         "parameters": {
             "type": "object",
             "properties": {
                 "max_price": {"type": "number", "description": "Maximum price in BRL"},
                 "body": {"type": "string", "enum": ["SUV", "sedan", "hatch"]},
             },
             "required": ["max_price"],
         }}},
    {"type": "function",
     "function": {
         "name": "car_details",
         "description": ("Get all the details of one car: trunk size (liters), fuel consumption (km/l), "
                         "color and city. Use it with a car_id returned by search_cars."),
         "parameters": {
             "type": "object",
             "properties": {"car_id": {"type": "string", "description": "The car id, like c082"}},
             "required": ["car_id"],
         }}},
    {"type": "function",
     "function": {
         "name": "compare_cars",
         "description": ("Compare 2 to 4 cars side by side (year, km, price, transmission, trunk size, consumption). "
                         "Use it when the user wants to choose between specific cars."),
         "parameters": {
             "type": "object",
             "properties": {"car_ids": {"type": "array", "items": {"type": "string"},
                                        "description": 'The car ids, like ["c082", "c109"]'}},
             "required": ["car_ids"],
         }}},
]

# O nome que o modelo vai usar → a função Python que NÓS vamos executar.
FUNCTIONS = {"search_cars": search_cars, "car_details": car_details, "compare_cars": compare_cars}

show(TOOLS[0])

{
  "type": "function",
  "function": {
    "name": "search_cars",
    "description": "Search the used-car stock by budget and body type. Returns how many cars matched and the 5 newest ones (car_id, model, year, km, price).",
    "parameters": {
      "type": "object",
      "properties": {
        "max_price": {
          "type": "number",
          "description": "Maximum price in BRL"
        },
        "body": {
          "type": "string",
          "enum": [
            "SUV",
            "sedan",
            "hatch"
          ]
        }
      },
      "required": [
        "max_price"
      ]
    }
  }
}


## 3.3 · Volta 1: o modelo **pede** uma ferramenta

Mesmas duas mensagens do Exemplo 1. A única diferença é o argumento `tools=TOOLS`.

🔮 **Antes de rodar.** O que vai estar em `content`, o texto da resposta?

In [8]:
messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": PEDIDO}]

resp = llm(messages, tools=TOOLS)          # ← agora com as ferramentas
msg = resp["message"]

print("📥 O QUE O MODELO RESPONDEU (volta 1)\n")
show(msg)

📥 O QUE O MODELO RESPONDEU (volta 1)

{
  "role": "assistant",
  "content": "",
  "tool_calls": [
    {
      "id": "call_d5sio9v3",
      "function": {
        "index": 0,
        "name": "search_cars",
        "arguments": {
          "max_price": 80000,
          "body": "SUV"
        }
      }
    }
  ]
}


### 🔍 O que aconteceu

- **`content` está vazio.** O modelo não respondeu ao usuário. Ele respondeu ao **nosso programa**.
- **`tool_calls` traz um pedido em JSON**, `search_cars` com `max_price: 80000` e `body: "SUV"`. "Até uns 80 mil" virou um número e "SUV" virou um valor do `enum`. Essa tradução de linguagem natural para argumentos foi o modelo que fez.
- **Nada foi executado.** O pedido está num dicionário Python, esperando alguém executá-lo. O modelo nunca executa nada.

## 3.4 · Nós executamos e devolvemos o resultado

Três passos, e os três são código nosso:

1. a decisão do modelo entra na conversa (`messages.append(msg)`);
2. executamos a função pedida, com os argumentos que ele escolheu;
3. o resultado volta para a conversa como uma mensagem de papel **`tool`**.

In [9]:
# 1. a decisão do modelo entra na conversa
messages.append(msg)

# 2. NÓS executamos a função pedida, com os argumentos que o modelo escolheu
call = msg["tool_calls"][0]["function"]
result = FUNCTIONS[call["name"]](**call["arguments"])
print(f"🔧 executamos {call['name']}({json.dumps(call['arguments'], ensure_ascii=False)})\n")
show(result)

# 3. o resultado volta para a conversa como uma mensagem de papel "tool"
messages.append({"role": "tool", "tool_name": call["name"],
                 "content": json.dumps(result, ensure_ascii=False)})

print("\n📜 A CONVERSA AGORA")
show_messages(messages)

🔧 executamos search_cars({"max_price": 80000, "body": "SUV"})

{
  "total_found": 22,
  "cars": [
    {
      "car_id": "c082",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 56500,
      "price_brl": 75000,
      "transmission": "automatic"
    },
    {
      "car_id": "c081",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 67700,
      "price_brl": 71000,
      "transmission": "manual"
    },
    {
      "car_id": "c109",
      "brand": "Volkswagen",
      "model": "Nivus",
      "year": 2021,
      "km": 70300,
      "price_brl": 75500,
      "transmission": "manual"
    },
    {
      "car_id": "c061",
      "brand": "Jeep",
      "model": "Renegade",
      "year": 2020,
      "km": 82700,
      "price_brl": 76500,
      "transmission": "manual"
    },
    {
      "car_id": "c003",
      "brand": "Chevrolet",
      "model": "Tracker",
      "year": 2019,
      "km": 91000,
      "price_brl": 74500,
      "transm

### 🔍 O que aconteceu

A busca encontrou 22 SUVs e devolveu os 5 mais novos. A conversa, que tinha 2 mensagens, agora tem 4: o pedido do usuário, a decisão do modelo (`[2]`) e o resultado da ferramenta (`[3]`). **Todo o estado do agente está nessa lista.** O modelo não guarda nada entre uma chamada e outra; na próxima, ele relê tudo.

## 3.5 · Volta 2: o modelo lê o resultado e decide de novo

Mesma chamada da volta 1, agora com a conversa de 4 mensagens.

🔮 **Antes de rodar.** Ele vai pedir outra ferramenta ou responder ao usuário?

In [10]:
resp = llm(messages, tools=TOOLS)
msg = resp["message"]
messages.append(msg)

print("📥 O QUE O MODELO RESPONDEU (volta 2)")
print("   pediu ferramenta?", bool(msg.get("tool_calls")), "\n")
print(msg["content"])

print("\n📜 A CONVERSA COMPLETA")
show_messages(messages)

📥 O QUE O MODELO RESPONDEU (volta 2)
   pediu ferramenta? False 

Com base nos resultados da pesquisa, aqui estão algumas opções de SUV que atendem aos seus critérios:

1. Renault Duster (car_id: c082) - Ano de 2021, com 56.500 km rodados, preço: R$ 75.000
2. Renault Duster (car_id: c081) - Ano de 2021, com 67.700 km rodados, preço: R$ 71.000
3. Volkswagen Nivus (car_id: c109) - Ano de 2021, com 70.300 km rodados, preço: R$ 75.500
4. Jeep Renegade (car_id: c061) - Ano de 2020, com 82.700 km rodados, preço: R$ 76.500
5. Chevrolet Tracker (car_id: c003) - Ano de 2019, com 91.000 km rodados, preço: R$ 74.500

Estes são os 5 SUVs mais recentes que atendem aos seus critérios de preço e modelo. Recomendo que você veja mais detalhes em cada um desses carros para tomar uma decisão. Você gostaria de saber mais sobre algum desses carros?

📜 A CONVERSA COMPLETA
[0] ⚙️  system    │ You are a used-car buying assistant in Natal, Brazil. Answer in the user's language. Use the tools to look up real li

### 🔍 O que aconteceu

Desta vez **não** há `tool_calls`. O modelo respondeu com texto, e é isso que encerra o loop. A resposta cita os `car_id` e começa por **`c082`, `c081` e `c109`**, o mesmo gabarito da Parte 1. Os fatos (ano, quilometragem, preço) vieram do resultado da ferramenta, e não da memória do modelo.

Foram **duas chamadas ao modelo** para uma única pergunta do usuário:

| Volta | Entrou no modelo | Saiu do modelo | O que nós fizemos |
|---|---|---|---|
| 1 | sistema + pedido + descrição das ferramentas | um pedido de `search_cars` em JSON | executamos a busca e devolvemos o resultado |
| 2 | tudo isso + a decisão + o resultado | texto para o usuário | nada: o loop acabou |

## 3.6 · O loop completo

As voltas 1 e 2 foram o mesmo código, repetido. Um agente é esse código dentro de um `for`: **chamar o modelo → se ele pedir ferramentas, executar e devolver → repetir até ele responder com texto**. O `max_steps` é o limite de voltas, o único guardrail obrigatório.

A função imprime cada passo intermediário.

In [11]:
def summary(result):
    """Uma linha sobre o resultado de uma ferramenta, para o trace."""
    if "error" in result:
        return "erro: " + result["error"]
    if "total_found" in result:                                        # search_cars
        return f'{result["total_found"]} encontrados; devolvemos {[c["car_id"] for c in result["cars"]]}'
    if "best" in result:                                               # compare_cars
        return "melhor em cada critério: " + ", ".join(f"{k} {'/'.join(v)}" for k, v in result["best"].items())
    return (f'{result["brand"]} {result["model"]} {result["year"]}: porta-malas {result["trunk_l"]} L, '   # car_details
            f'consumo {result["consumption_kml"]} km/l, {result["color"]}, {result["city"]}')

def run_agent(pedido, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pedido}]
    print(f"👤 {pedido}\n")
    for volta in range(1, max_steps + 1):
        print(f"── volta {volta} " + "─" * 60)
        msg = llm(messages, tools=TOOLS)["message"]          # REASON: o modelo decide
        messages.append(msg)
        if not msg.get("tool_calls"):                        # texto em vez de ferramenta: fim do loop
            print("🤖 o modelo respondeu com texto → fim do loop\n")
            print(msg["content"])
            return messages
        for c in msg["tool_calls"]:                          # o modelo pode pedir várias de uma vez
            name, args = c["function"]["name"], c["function"]["arguments"]
            print(f"🤖 o modelo pediu   → {name}({json.dumps(args, ensure_ascii=False)})")
            try:
                result = FUNCTIONS[name](**args)             # ACT: nós executamos
            except Exception as e:                           # um erro vira observação, não derruba o loop
                result = {"error": f"{type(e).__name__}: {e}"}
            print(f"🔧 nós executamos   → {summary(result)}")
            messages.append({"role": "tool", "tool_name": name,       # OBSERVE: o resultado volta
                             "content": json.dumps(result, ensure_ascii=False)})
    print(f"(parei depois de {max_steps} voltas sem uma resposta final)")
    return messages

conversa = run_agent(PEDIDO)

👤 Quero um SUV para a família, até uns 80 mil. O que você recomenda?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → search_cars({"max_price": 80000, "body": "SUV"})
🔧 nós executamos   → 22 encontrados; devolvemos ['c082', 'c081', 'c109', 'c061', 'c003']
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

Com base nos resultados da pesquisa, aqui estão algumas opções de SUV que atendem aos seus critérios:

1. Renault Duster (car_id: c082) - Ano de 2021, com 56.500 km rodados, preço: R$ 75.000
2. Renault Duster (car_id: c081) - Ano de 2021, com 67.700 km rodados, preço: R$ 71.000
3. Volkswagen Nivus (car_id: c109) - Ano de 2021, com 70.300 km rodados, preço: R$ 75.500
4. Jeep Renegade (car_id: c061) - Ano de 2020, com 82.700 km rodados, preço: R$ 76.500
5. Chevrolet Tracker (car_id: c003) - Ano de 2019, com 91.000 km rodados, preço: R$ 74.500

Estes são os 5 SUVs mais recentes

### 🔍 O que aconteceu

O mesmo resultado das seções 3.3 a 3.5, agora dentro de um `for`: uma volta para pedir a busca, outra para responder. Tirando os `print`, `run_agent` tem umas quinze linhas. **Esse é o agente inteiro.**

Agora uma pergunta para cada uma das outras duas ferramentas. A primeira pede a cidade e a cor de um carro, que só a `car_details` tem. A segunda pede para comparar dois carros. Em nenhuma das duas o pedido diz qual ferramenta usar; o modelo escolhe lendo as descrições.

In [12]:
conversa = run_agent("Em que cidade está o c109 e qual é a cor dele?")
print("\n" + "═" * 72 + "\n")
conversa = run_agent("Compara o c082 com o c109 pra mim. Qual tem o maior porta-malas e qual consome menos?")

👤 Em que cidade está o c109 e qual é a cor dele?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → car_details({"car_id": "c109"})
🔧 nós executamos   → Volkswagen Nivus 2021: porta-malas 415 L, consumo 12.5 km/l, vermelho, Natal
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

O carro com o car_id c109 é da marca Volkswagen, modelo Nivus, e é um SUV. Ele está na cidade de Natal. A cor desse carro é vermelho.

════════════════════════════════════════════════════════════════════════

👤 Compara o c082 com o c109 pra mim. Qual tem o maior porta-malas e qual consome menos?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → compare_cars({"car_ids": ["c082", "c109"]})
🔧 nós executamos   → melhor em cada critério: largest_trunk c082, most_economical c109, cheapest c082, newest c082/c109, lowest_km c082
── volta 2 ───────────────────────────

### 🔍 O que aconteceu

Cada pergunta levou o modelo a uma ferramenta diferente. Na primeira, ele pediu `car_details("c109")` e respondeu que o Nivus está em Natal e é vermelho. Na segunda, pediu `compare_cars(["c082", "c109"])`. Repare no trace da comparação: a ferramenta já devolveu o vencedor de cada critério, maior porta-malas `c082` e mais econômico `c109`, e o modelo só precisou ler e redigir.

A frase do modelo pode sair torta. Durante a preparação, ele escreveu que o Nivus "consome mais gasolina, com 12,5 km/l" e, em seguida, "portanto o c109 é mais econômico". A conclusão está certa porque veio do `best` da ferramenta; a justificativa, não. Confira com o próprio estoque:

In [13]:
cars[cars.car_id.isin(["c082", "c109"])][["car_id", "brand", "model", "year", "trunk_l", "consumption_kml", "color", "city"]]

,car_id,brand,model,year,trunk_l,consumption_kml,color,city
81,c082,Renault,Duster,2021,475,10.4,cinza,Macaíba
108,c109,Volkswagen,Nivus,2021,415,12.5,vermelho,Natal


## 3.7 · Quando o loop dá mais de uma volta

Às vezes o modelo precisa do resultado de uma ferramenta para decidir a próxima. Para comparar "os dois mais novos", ele primeiro precisa buscar para saber **quais** são (volta 1) e só depois comparar esses dois (volta 2).

In [14]:
conversa = run_agent("Quero um SUV até 80 mil para a família. Compare os dois mais novos e me diga qual você escolheria.")

print("\n📜 A CONVERSA COMPLETA")
show_messages(conversa)

👤 Quero um SUV até 80 mil para a família. Compare os dois mais novos e me diga qual você escolheria.

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → search_cars({"max_price": 80000, "body": "SUV"})
🔧 nós executamos   → 22 encontrados; devolvemos ['c082', 'c081', 'c109', 'c061', 'c003']
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → compare_cars({"car_ids": ["c082", "c081"]})
🔧 nós executamos   → melhor em cada critério: largest_trunk c081/c082, most_economical c081, cheapest c081, newest c081/c082, lowest_km c082
── volta 3 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

Os dois SUVs mais novos disponíveis são:
1. Renault Duster com car_id c081, de 2021, com 67,700 km e preço de R$ 71,000.
2. Renault Duster com car_id c082, de 2021, com 56,500 km e preço de R$ 75,000.

Comparando esses dois modelos:
- O Renault Duster c081 tem maior capa

### 🔍 O que aconteceu

Três voltas. Na volta 1, `search_cars`. Na volta 2, `compare_cars(["c082", "c081"])`, com os dois ids que **o próprio modelo** tirou do resultado da busca. Na volta 3, a resposta. É o ReAct do slide: raciocinar, agir, observar e raciocinar de novo com base no que observou. A conversa terminou com 7 mensagens: duas decisões, dois resultados e a resposta.

Agora confira a resposta com o trace. Os dois Duster são de 2021 e têm 475 litros de porta-malas, e a ferramenta marcou esses dois critérios como **empate** (`c081/c082`). Durante a preparação, o modelo escreveu que o `c081` "tem o maior porta-malas" e "é o mais novo", o que é uma simplificação. A escolha final, o `c081`, é defensável: ele é mais barato e mais econômico, e esses dois critérios a ferramenta deu sem empate. A lição é a mesma da comparação anterior: os números vêm das ferramentas, mas o texto é do modelo, e o texto se confere.

Modelos de 3 bilhões de parâmetros também oscilam em pedidos assim. Em outras formulações, o modelo às vezes para depois da busca, achando que já tem o suficiente. Por isso um agente se **avalia** com muitos pedidos, rodados várias vezes, e não com uma execução. É a fase 3 do Trabalho 2.

---
# Parte 4 · Resumo

```
👤 pedido
   │
   ▼
🤖 modelo ──► pede uma ferramenta (JSON) ──► 🔧 NÓS executamos ──► o resultado volta como mensagem "tool"
   ▲                                                                          │
   └──────────────────────────────── repete ◄────────────────────────────────┘
   │
   ▼  (quando o modelo responde com texto)
💬 resposta
```

1. **O modelo nunca executa nada.** Ele escreve um pedido em JSON, e o nosso código decide executar.
2. **Uma ferramenta tem nome, descrição e parâmetros.** É isso que o modelo lê para decidir o que pedir.
3. **Todo o estado do agente é a lista `messages`.** A cada volta entram a decisão do modelo e o resultado da ferramenta, e a cada chamada o modelo relê tudo.
4. **O loop termina quando o modelo responde com texto**, ou quando as voltas acabam (`max_steps`).

### Daqui para a fase 1 do Trabalho 2

Este notebook é o esqueleto da fase 1. Troque carros pelo domínio do seu grupo e implemente as três ferramentas do contrato (`search`, `details` e `compare`), com um dataset de 200 a 1000 itens, uma persona e uma conversa de exemplo. **Prazo: 04/11, no início da aula.**

| Você viu aqui | Volta como |
|---|---|
| A descrição de uma ferramenta (3.2) | Um servidor MCP, aula 11 |
| O gabarito e a conferência pelo `car_id` | A avaliação de agentes, aula 12 |
| O loop com `max_steps` (3.6) | O *harness*, aula 16, e o Claude Code, aula 14 |